In [ ]:
import sys
sys.path.append('/home/potzschf/repos/')
from helperToolz.helpsters import *
from helperToolz.dicts_and_lists import *
from helperToolz.guzinski import * 
import geopandas as gpd
from collections import defaultdict
from joblib import Parallel, delayed
from datetime import datetime, timedelta
import time


# state the compositing scheme, the folder, where evap files were stored and number of cores to be used during parellel running parts
comps = ['minVZA']#['maxLST']
sharps = ['S2only']#['S2only']

years = [y for y in range(2018,2025,1)]

# tiles_to_process = get_forcetiles_range([file for file in getFilelist('/data/Aldhani/eoagritwin/et/Auxiliary/DEM/Force_Tiles/THUENEN/', '.tif') if '_2021_' in file])
tiles_to_process = [
    'X0053_Y0049',
    'X0053_Y0050',
    'X0070_Y0049',
    'X0064_Y0048',
    'X0071_Y0045',
    'X0063_Y0044',
    'X0064_Y0045'
    ]

for year in years:

    pathi = f"/data/Aldhani/eoagritwin/et/SharpEvap/ReferenceStations/{year}/evap/"#f"/data/Aldhani/eoagritwin/et/Sentinel3/LST/SharpEvap/ReferenceStations/{year}/evap/"
    
    if os.path.exists(pathi):
        pass
    else:
        continue
    
    ncores = 6

    single_soilFuncL = []
    single_canopyFuncL = []
    sum_FuncL = []
    mean_FuncL = []

    for tile_to_process in tiles_to_process:
        tile_to_process_ = ''.join(tile_to_process.split('_'))

        # check if tiles were processed at all
        if os.path.exists(f"{pathi}{tile_to_process_}/"):
            print(tile_to_process_)
        else:
            print(f"{tile_to_process_} not found!")
            continue

        # get the max extent for vrts. this is based on the mask (which actually was dervied from a vrt created and then deleted as there are some with smaller extent)
        path_to_agro = '/data/Aldhani/eoagritwin/et/Auxiliary/DEM/Force_Tiles/THUENEN/'
        thuenen = [file for file in getFilelist(path_to_agro, '.tif') if tile_to_process in file and f"_{year}_" in file][0] # if any tile name is in file
        
        if(len(thuenen)) == 0:
            thuenen = [file for file in getFilelist(path_to_agro, '.tif') if tile_to_process in file and '2021' in file][0]
        mask_ds = gdal.Open(thuenen)
        gt = mask_ds.GetGeoTransform()
        mask_proj = mask_ds.GetProjection()
        xmin = gt[0]
        ymax = gt[3]
        px_size_x = gt[1]
        px_size_y = abs(gt[5])
        xres = mask_ds.RasterXSize
        yres = mask_ds.RasterYSize
        xmax = xmin + xres * px_size_x
        ymin = ymax - yres * px_size_y

        vrt_options = gdal.BuildVRTOptions(
            outputBounds=[xmin, ymin, xmax, ymax],
            xRes=px_size_x,       
            yRes=px_size_y, 
            outputSRS=mask_proj,
            resampleAlg='nearest',
            separate=False
        )

        # create sub-folder
        for comp in comps:
            for sharp in sharps:
                # create sub-folder
                evap_outFolder = f"{pathi}{tile_to_process_}/{comp}/{sharp}/"
                vrt_folder= path_safe(f"{evap_outFolder}vrt/")

                # separate for 4 different evap products (soil vs canopy, ssrd calc vs ssrd func)
                all_files = [file for file in getFilelist(f"{pathi}{tile_to_process_}/", '.tif', deep=True) if comp in file and sharp in file]

                soil_func_files = [file for file in all_files if 'Soil_func' in file]
                #soil_calc_files = [file for file in all_files if 'Soil_calc' in file]

                canopy_func_files = [file for file in all_files if 'Canopy_func' in file]
                #canopy_calc_files = [file for file in all_files if 'Canopy_calc' in file]

                months = [REAL_INT_TO_MONTH[m] for m in range(1,13) if growingSeasonChecker(m)]

                date_reg = re.compile(r'20\d{2}_(?:' + '|'.join(months) + r')_\d{1,2}_')
                date_reg2 = re.compile(r'20\d{2}_(?:' + '|'.join(months) + r')_\d{1,2}.')

                filesList = [soil_func_files, canopy_func_files] # , soil_calc_files , canopy_calc_files
                filesNames = ['Soil_func', 'Canopy_func'] # , 'Soil_calc' , 'Canopy_calc'

                # create daily vrts mosaics for each type of evap output (soil, canopy)
                for fname, fileL in zip(filesNames, filesList):
                    dicti = defaultdict(list)
                    for file in fileL:
                        match = date_reg.search(file)
                        if match:
                            date = match.group()[:-1]  
                            year, month, day = date.split('_')
                            day = day.zfill(2)  # "1" → "01", "13" → "13"
                            date_key = f"{year}_{month}_{day}"  
                            dicti[date_key].append(file)
                            continue
                    
                    for key, files in dicti.items():
                        
                        outfolder = f'{vrt_folder}{fname}/'

                        vrt_path = path_safe(f'{outfolder}{fname}_{key}_{comp}_{sharp}.vrt')
                        vrt = gdal.BuildVRT(vrt_path, files, options=vrt_options)
                        vrt = None

                        convertVRTpathsTOrelative(vrt_path=vrt_path)

                # load the comp dates incorporate a sanity check maybe???
                if len(set([len(pd.read_csv(file)['compdates'].tolist()) for file in getFilelist(f"{pathi}{tile_to_process_}/", '.csv', deep=True)])) > 1:
                    raise ValueError('There are two different processing periods inherent in the processed files - this causes troubles')
                else:
                    compDates = [int(pd.read_csv(file)['compdates'][0]) for file in getFilelist(f"{pathi}{tile_to_process_}/", '.csv', deep=True)]

                starts = [datetime.strptime(str(compDate), '%Y%m%d') - timedelta(days=4) for compDate in compDates]
                ends = [datetime.strptime(str(compDate), '%Y%m%d') + timedelta(days=4) for compDate in compDates]

                # check lowest and highest start date in vrts and adapt start and end if needed
                vrt_dates = list(dicti.keys())
                vrt_datesL = []
                for vrt_date in vrt_dates:
                    year, month, day = vrt_date.split('_') 
                    vrt_datesL.append(f'{year}{MONTH_TO_02D[month]}{day}')
                vrt_start = datetime.strptime(str(min(vrt_datesL)), '%Y%m%d')
                vrt_end = datetime.strptime(str(max(vrt_datesL)), '%Y%m%d')

                print(vrt_start)
                print(starts)
                print(compDates)
                if vrt_start < starts[0]:
                    starts[0] = vrt_start

                if vrt_end > ends[-1]:
                    ends[-1] = vrt_end

                # load mask for vrts (based on the polygonized delineated fields (@20m!!!!))
                mask_arr = mask_ds.GetRasterBand(1).ReadAsArray()
                mask_arr[mask_arr>0] = 1

                outpath_single = path_safe(f"{evap_outFolder}median_9day/ET_single/")
                outpath_sum = path_safe(f"{evap_outFolder}median_9day/ET_sum/")

                joblist = []

                for et_var in ['Soil_func', 'Canopy_func']: # , 'Soil_calc', 'Canopy_calc'
                    for date_start, date_end in zip(starts, ends):
                        joblist.append([vrt_folder, et_var, date_start, date_end, date_reg2, outpath_single, mask_arr])
                print(f'\n{len(joblist)} jobs will be processed\n')


                if __name__ == '__main__':
                    starttime = time.strftime("%a, %d %b %Y %H:%M:%S", time.localtime())
                    print("--------------------------------------------------------")
                    print("Starting process, time:" + starttime)
                    print("")

                    Parallel(n_jobs=ncores)(delayed(make_ET_median)(job[0], job[1], job[2], job[3], job[4], job[5], job[6]) for job in joblist)

                    print("")
                    endtime = time.strftime("%a, %d %b %Y %H:%M:%S", time.localtime())
                    print("--------------------------------------------------------")
                    print("--------------------------------------------------------")
                    print("start : " + starttime)
                    print("end: " + endtime)
                    print("")


                print('all median composites calculated - now canopy and soil will be sumed up')

                medList = getFilelist(outpath_single, '.tif')

                for date_start in starts:

                    dt = datetime.strftime(date_start, '%Y_%m_%d')
                    #date_sub_calc = [med_arr for med_arr in medList if dt in med_arr and 'calc' in med_arr]
                    date_sub_func = [med_arr for med_arr in medList if dt in med_arr and 'func' in med_arr]

                    [single_soilFuncL.append(soil_file) for soil_file in date_sub_func if 'Soil_' in soil_file]
                    [single_canopyFuncL.append(canopy_file) for canopy_file in date_sub_func if 'Canopy_' in canopy_file]
                    # arrL = []
                    # for pathi in date_sub_calc:
                    #     ds = gdal.Open(pathi)
                    #     arrL.append(ds.GetRasterBand(1).ReadAsArray())
                    # arr_sum = np.nansum(np.dstack([arrL[0], arrL[1]]),axis=2)
                    # arr_sum[arr_sum >= 10] = np.nan
                    # # we cut off the highest values at the Polish border for now
                    # cutOff = np.nanpercentile(arr_sum, [99.9])[0]
                    # arr_sum[arr_sum > cutOff] = np.nan
                    # makeTif_np_to_matching_tif(arr_sum, pathi, f"{outpath_sum}{dt}_median_ET_calc.tif", 0)

                    arrL = []
                    for file_path in date_sub_func:
                        ds = gdal.Open(file_path)
                        arrL.append(ds.GetRasterBand(1).ReadAsArray())
                    arr_sum = np.nansum(np.dstack([arrL[0], arrL[1]]),axis=2)
                    arr_sum[arr_sum >= 10] = np.nan

                    arr_mean = np.nanmean(np.dstack([arrL[0], arrL[1]]),axis=2)
                    arr_mean[arr_mean >= 10] = np.nan
                    # we cut off the highest values at the Polish border for now
                    cutOff = np.nanpercentile(arr_sum, [99.9])[0]
                    arr_sum[arr_sum > cutOff] = np.nan
                    makeTif_np_to_matching_tif(arr_sum, file_path, f"{outpath_sum}{dt}_median_ET_sum_func_{comp}_{sharp}.tif", 0)
                    makeTif_np_to_matching_tif(arr_mean, file_path, f"{outpath_sum}{dt}_median_ET_mean_func_{comp}_{sharp}.tif", 0)

                    sum_FuncL.append(f"{outpath_sum}{dt}_median_ET_sum_func_{comp}_{sharp}.tif")
                    mean_FuncL.append(f"{outpath_sum}{dt}_median_ET_mean_func_{comp}_{sharp}.tif")


    # build overlord vrts
    for comp in comps:
        for sharp in sharps:
            
            sing_soil = [f1 for f1 in single_soilFuncL if comp in f1 and sharp in f1]
            cdates = list(set([sing_s.split('Soil_func_')[-1].split('_median')[0] for sing_s in sing_soil]))
    
            for cdate in cdates:
                vrtL = [sing_s for sing_s in sing_soil if cdate in sing_s]
        
                vrt_path = path_safe(path_safe(f"{pathi}vrt/{comp}/{sharp}/ET_Soil_func_{cdate}_{comp}_{sharp}.vrt"))#f'{outfolder}{fname}_{key}_{comp}_{sharp}.vrt')
                vrt = gdal.BuildVRT(vrt_path, vrtL)
                vrt = None
                convertVRTpathsTOrelative(vrt_path=vrt_path)
            
            sing_canop = [f1 for f1 in single_canopyFuncL if comp in f1 and sharp in f1]
            cdates = list(set([sing_c.split('Canopy_func_')[-1].split('_median')[0] for sing_c in sing_canop]))

            for cdate in cdates:
                vrtL = [sing_c for sing_c in sing_canop if cdate in sing_c]
            
                vrt_path = path_safe(path_safe(f"{pathi}vrt/{comp}/{sharp}/ET_Canopy_func_{cdate}_{comp}_{sharp}.vrt"))#f'{outfolder}{fname}_{key}_{comp}_{sharp}.vrt')
                vrt = gdal.BuildVRT(vrt_path, vrtL)
                vrt = None
                convertVRTpathsTOrelative(vrt_path=vrt_path)

            sumi = [f1 for f1 in sum_FuncL if comp in f1 and sharp in f1]
            cdates = list(set([summi.split('/')[-1].split('_median_ET')[0] for summi in sumi]))

            for cdate in cdates:
                vrtL = [summi for summi in sumi if cdate in summi]

                vrt_path = path_safe(path_safe(f"{pathi}vrt/{comp}/{sharp}/ET_Soil_and_Canopy_sum_func_{cdate}_{comp}_{sharp}.vrt"))#f'{outfolder}{fname}_{key}_{comp}_{sharp}.vrt')
                vrt = gdal.BuildVRT(vrt_path, vrtL)
                vrt = None
                convertVRTpathsTOrelative(vrt_path=vrt_path)

            meani = [f1 for f1 in mean_FuncL if comp in f1 and sharp in f1]
            cdates = list(set([meanni.split('/')[-1].split('_median_ET')[0] for meanni in meani]))

            for cdate in cdates:
                vrtL = [meanni for meanni in meani if cdate in meanni]

                vrt_path = path_safe(path_safe(f"{pathi}vrt/{comp}/{sharp}/ET_Soil_and_Canopy_mean_func_{cdate}_{comp}_{sharp}.vrt"))#f'{outfolder}{fname}_{key}_{comp}_{sharp}.vrt')
                vrt = gdal.BuildVRT(vrt_path, vrtL)
                vrt = None
                convertVRTpathsTOrelative(vrt_path=vrt_path)